# DS_Day01_34 — E-Commerce Customer Segmentation
## Who Are Our Customers?

Synthetic-data project using RFM analysis and KNN.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df=pd.read_csv("ecommerce_customer_segmentation_dataset.csv")
df.head()

## 1. Data Cleaning and Validation

In [ ]:
print("Shape:",df.shape)
print("Missing values:\n",df.isnull().sum())
print("Duplicate rows:",df.duplicated().sum())
display(df.describe())

In [ ]:
df["Spending"]=df["Spending"].fillna(df["Spending"].median())
df["Product_Diversity"]=df["Product_Diversity"].fillna(df["Product_Diversity"].median())
df["Discount_Usage"]=df["Discount_Usage"].fillna(df["Discount_Usage"].median())
df=df.drop_duplicates().reset_index(drop=True)
print(df.isnull().sum())

## 2. RFM Calculation

In [ ]:
df["R_Score"]=pd.qcut(df["Recency"],5,labels=[5,4,3,2,1],duplicates="drop").astype(int)
df["F_Score"]=pd.qcut(df["Purchase_Frequency"].rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
df["M_Score"]=pd.qcut(df["Spending"].rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
df["RFM_Score"]=df[["R_Score","F_Score","M_Score"]].sum(axis=1)
display(df[["Customer_ID","R_Score","F_Score","M_Score","RFM_Score"]].head())

## 3. Behaviour-Based Segmentation

In [ ]:
def segment(r):
    if r.R_Score>=4 and r.F_Score>=4 and r.M_Score>=4: return "High Value"
    if r.F_Score>=4 and r.M_Score<=2: return "Frequent Low Spend"
    if r.M_Score>=4 and r.F_Score<=2: return "Heavy Infrequent"
    if r.R_Score<=2 and r.F_Score<=2 and r.M_Score<=2: return "At Risk / Low Value"
    return "Loyal / Potential"
df["Customer_Segment"]=df.apply(segment,axis=1)
display(df.Customer_Segment.value_counts())

## 4. Four Required Visualisations

In [ ]:
import seaborn as sns
plt.figure(figsize=(8,5)); sns.countplot(data=df,x="Customer_Segment"); plt.title("Customer Segment Distribution"); plt.xticks(rotation=25); plt.show()
plt.figure(figsize=(8,5)); sns.scatterplot(data=df,x="Purchase_Frequency",y="Spending",hue="Customer_Segment"); plt.title("Purchase Frequency vs Spending"); plt.show()
plt.figure(figsize=(8,5)); sns.scatterplot(data=df,x="Recency",y="Spending",hue="Customer_Segment"); plt.title("Recency vs Spending"); plt.show()
plt.figure(figsize=(8,5)); sns.barplot(data=df,x="Customer_Segment",y="Spending",errorbar=None); plt.title("Average Spending by Customer Segment"); plt.xticks(rotation=25); plt.show()

## 5. Segment Comparison

In [ ]:
summary=df.groupby("Customer_Segment").agg(
Customers=("Customer_ID","count"),Avg_Recency=("Recency","mean"),
Avg_Frequency=("Purchase_Frequency","mean"),Avg_Spending=("Spending","mean"),
Avg_Diversity=("Product_Diversity","mean"),Avg_Discount=("Discount_Usage","mean")
).round(2)
display(summary)

freq_low=df[(df.Purchase_Frequency>=df.Purchase_Frequency.quantile(.75))&(df.Spending<=df.Spending.quantile(.25))]
heavy_inf=df[(df.Spending>=df.Spending.quantile(.75))&(df.Purchase_Frequency<=df.Purchase_Frequency.quantile(.25))]
print("Frequent but low-spending:",len(freq_low))
print("Heavy-spending but infrequent:",len(heavy_inf))

## 6. KNN Classification

In [ ]:
features=["Recency","Purchase_Frequency","Spending","Product_Diversity","Discount_Usage"]
X=df[features]; y=df.Customer_Segment
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
knn=Pipeline([("scaler",StandardScaler()),("knn",KNeighborsClassifier(n_neighbors=7))])
knn.fit(Xtr,ytr); pred=knn.predict(Xte)
print("Accuracy:",round(accuracy_score(yte,pred),3))
print("Precision:",round(precision_score(yte,pred,average="weighted",zero_division=0),3))
print("Recall:",round(recall_score(yte,pred,average="weighted",zero_division=0),3))
print("F1:",round(f1_score(yte,pred,average="weighted",zero_division=0),3))
print(classification_report(yte,pred,zero_division=0))

## 7. Nearest-Behaviour Anomaly Flagging

In [ ]:
zs=StandardScaler().fit_transform(df[features])
nn=NearestNeighbors(n_neighbors=6).fit(zs)
dist,_=nn.kneighbors(zs)
df["Nearest_Behaviour_Distance"]=dist[:,1:].mean(axis=1)
threshold=df.Nearest_Behaviour_Distance.quantile(.95)
df["Behaviour_Anomaly"]=df.Nearest_Behaviour_Distance>threshold
print("Anomaly threshold:",round(threshold,3))
print("Flagged records:",int(df.Behaviour_Anomaly.sum()))
display(df[df.Behaviour_Anomaly].sort_values("Nearest_Behaviour_Distance",ascending=False).head(20))

## 8. Key Insights and Marketing Action Plan

1. RFM analysis separates customers into behaviour-based groups.
2. High-value customers combine recent purchases, frequent activity and high spending.
3. Frequent low-spending customers are opportunities to increase basket size.
4. Heavy-spending but infrequent customers are reactivation opportunities.
5. At-risk/low-value customers need selective win-back campaigns.
6. Product diversity and discount usage add targeting context.
7. KNN identifies similar behaviour patterns; nearest-neighbour distance flags unusual patterns.

**Actions:** VIP rewards for High Value; cross-sell and loyalty benefits for Loyal/Potential; bundles for Frequent Low Spend; personalized reactivation for Heavy Infrequent; low-cost win-back for At Risk/Low Value; review anomalous records before automated action.

## 9. Conclusion
RFM-based segmentation provides a behaviour-focused view of customers. KNN classification and nearest-neighbour distance add a similarity-based machine-learning layer that can support targeted marketing and unusual-pattern review.

In [ ]:
df.to_csv("ecommerce_customer_segmentation_cleaned.csv",index=False)\nprint("Saved cleaned dataset.")